In [1]:
from PIL import Image

In [2]:
img_list = ["/Users/evillz/Downloads/cell0.png", 
            "/Users/evillz/Downloads/cell1.png",
            "/Users/evillz/Downloads/cell2.png",
            "/Users/evillz/Downloads/cell3.png",
            "/Users/evillz/Downloads/cell4.png"]


In [4]:
# create a transparent GIF from img_list
output_path = "/Users/evillz/Downloads/animation_tether_V2.gif"

# open images and ensure RGBA
imgs = [Image.open(p).convert("RGBA") for p in img_list]

# canvas size (max width/height)
size = (max(im.width for im in imgs), max(im.height for im in imgs))

# pad/center each frame onto a transparent RGBA canvas
frames_rgba = []
for im in imgs:
    canvas = Image.new("RGBA", size, (0, 0, 0, 0))
    x = (size[0] - im.width) // 2
    y = (size[1] - im.height) // 2
    canvas.paste(im, (x, y), im)
    frames_rgba.append(canvas)

# build a palette from the first frame but force inclusion of a unique color for transparency
w, h = size
tmp = Image.new("RGBA", (w + 1, h + 1), (0, 0, 0, 0))
tmp.paste(frames_rgba[0], (0, 0), frames_rgba[0])
unique_color = (255, 0, 255, 255)  # magenta marker
tmp.putpixel((w, h), unique_color)
palette_image = tmp.convert("P", palette=Image.ADAPTIVE, colors=256)

# find palette index of the unique color (will be used as transparency index)
transparency_index = palette_image.getpixel((w, h))

# quantize all frames using the same palette
pal_frames = []
for f in frames_rgba:
    filled = Image.new("RGBA", size, unique_color)
    filled.alpha_composite(f)
    pal_frames.append(filled.convert("RGB").quantize(palette=palette_image, dither=Image.Dither.FLOYDSTEINBERG))

# save GIF with transparency
pal_frames[0].save(
    output_path,
    save_all=True,
    append_images=pal_frames[1:],
    duration=1000,
    loop=0,
    transparency=transparency_index,
    disposal=2
)